In [22]:
!pip install biopython

In [23]:
import sys, numpy, torch
print(sys.executable)
print("Python:", sys.version)
print("NumPy:", numpy.__version__)
print("Torch:", torch.__version__)


/Users/piyushagrawal/miniconda/bin/python
Python: 3.12.2 | packaged by conda-forge | (main, Feb 16 2024, 21:00:12) [Clang 16.0.6 ]
NumPy: 1.26.4
Torch: 2.2.2


In [67]:
####### Import

import numpy as np
import torch
import esm
from tqdm.auto import tqdm

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.metrics import roc_auc_score, matthews_corrcoef, accuracy_score

from Bio.SeqUtils.ProtParam import ProteinAnalysis
import pandas as pd

In [80]:
###### Load your data (exactly your files)

VALID_AA = set("ACDEFGHIKLMNPQRSTVWY")

def load_and_clean(fname):
    clean, bad = [], []
    with open(fname) as f:
        for line in f:
            seq = line.strip().upper()
            if not seq:
                continue
            if set(seq).issubset(VALID_AA):
                clean.append(seq)
            else:
                bad.append(seq)
    return clean, bad

pos, bad_pos = load_and_clean("unique_peptide_pos")
neg, bad_neg = load_and_clean("unique_peptide_random_neg")
marine, bad_marine = load_and_clean("unique_peptide_marine")

print("POS kept / removed:", len(pos), "/", len(bad_pos))
print("NEG kept / removed:", len(neg), "/", len(bad_neg))
print("MARINE kept / removed:", len(marine), "/", len(bad_marine))

POS kept / removed: 1224 / 12
NEG kept / removed: 1224 / 12
MARINE kept / removed: 1224 / 12


In [81]:
####### Length filtering (IMPORTANT for ACPs)

def length_filter(seqs, min_len=10, max_len=50):
    return [s for s in seqs if min_len <= len(s) <= max_len]

pos = length_filter(pos)
neg = length_filter(neg)
marine = length_filter(marine)

print("After length filter:")
print("POS:", len(pos))
print("NEG:", len(neg))
print("MARINE:", len(marine))

After length filter:
POS: 1076
NEG: 1076
MARINE: 1076


In [82]:
#### Build dataset + split (DEFINES X_train)

X = pos + neg
y = np.array([1]*len(pos) + [0]*len(neg))

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

print("Train:", len(X_train))
print("Test :", len(X_test))


Train: 1721
Test : 431


In [83]:
#### HARD SAFETY CHECK (PREVENTS TOKEN ERRORS)

def assert_clean(seqs):
    for s in seqs:
        if not set(s).issubset(VALID_AA):
            raise ValueError(f"Invalid sequence still present: {s}")

assert_clean(X_train)
assert_clean(X_test)
assert_clean(marine)

print("✔ All sequences valid")

✔ All sequences valid


In [84]:
####### Compute physicochemical features (CORE IMPROVEMENT)

from Bio.SeqUtils.ProtParam import ProteinAnalysis
import numpy as np

def aliphatic_index(seq):
    seq = seq.upper()
    length = len(seq)
    return (
        (seq.count("A") / length) * 100 +
        2.9 * (seq.count("V") / length) * 100 +
        3.9 * ((seq.count("I") + seq.count("L")) / length) * 100
    )

def compute_physchem(seqs):
    feats = []
    for s in seqs:
        pa = ProteinAnalysis(s)
        feats.append([
            len(s),
            pa.charge_at_pH(7.0),
            pa.gravy(),
            pa.aromaticity(),
            pa.instability_index(),
            aliphatic_index(s),
            pa.isoelectric_point()
        ])
    return np.array(feats)

phys_tr = compute_physchem(X_train)
phys_te = compute_physchem(X_test)
phys_mar = compute_physchem(marine)

print(phys_tr.shape, phys_te.shape, phys_mar.shape)

(1721, 7) (431, 7) (1076, 7)


In [85]:
#### Scale physicochemical features

from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
phys_tr = scaler.fit_transform(phys_tr)
phys_te = scaler.transform(phys_te)
phys_mar = scaler.transform(phys_mar)

In [86]:
##### Load ESM2 (SAFE LOCAL MODEL)

device = "cpu"

model, alphabet = esm.pretrained.esm2_t6_8M_UR50D()
batch_converter = alphabet.get_batch_converter()
model.eval()

print("✔ ESM2 loaded")

✔ ESM2 loaded


In [87]:
###### Memory-safe embedding function (DO NOT MODIFY)

def esm2_embed(seqs, batch_size=8):
    embs = []
    for i in tqdm(range(0, len(seqs), batch_size)):
        batch = seqs[i:i+batch_size]
        data = [(str(j), s) for j, s in enumerate(batch)]
        _, _, tokens = batch_converter(data)

        with torch.no_grad():
            out = model(tokens, repr_layers=[6])
            reps = out["representations"][6]

        for k, seq in enumerate(batch):
            embs.append(reps[k, 1:len(seq)+1].mean(0).numpy())

        del tokens, out, reps
    return np.vstack(embs)

In [88]:
##### Generate embeddings

Xtr_esm = esm2_embed(X_train)

Xte_esm = esm2_embed(X_test)
Xmar_esm = esm2_embed(marine)

  0%|          | 0/216 [00:00<?, ?it/s]

  0%|          | 0/54 [00:00<?, ?it/s]

  0%|          | 0/135 [00:00<?, ?it/s]

In [89]:
###### Concatenate ESM2 + PhysChem

Xtr = np.hstack([Xtr_esm, phys_tr])
Xte = np.hstack([Xte_esm, phys_te])
Xmar = np.hstack([Xmar_esm, phys_mar])

print(Xtr.shape)

(1721, 327)


In [90]:
##### Train models (SVM + LR)

lr = LogisticRegression(max_iter=5000, class_weight="balanced")
lr.fit(Xtr, y_train)

prob_lr = lr.predict_proba(Xte)[:,1]
pred_lr = (prob_lr >= 0.5).astype(int)

print("LR AUC:", roc_auc_score(y_test, prob_lr))
print("LR MCC:", matthews_corrcoef(y_test, pred_lr))

LR AUC: 0.8907622739018088
LR MCC: 0.6156041944292964


In [91]:
##### SVM (Usually best for ACP)

svm = SVC(
    kernel="linear",
    C=2.0,
    gamma="scale",
    probability=True,
    class_weight="balanced"
)

svm.fit(Xtr, y_train)

prob_svm = svm.predict_proba(Xte)[:,1]
pred_svm = (prob_svm >= 0.5).astype(int)

print("SVM AUC:", roc_auc_score(y_test, prob_svm))
print("SVM MCC:", matthews_corrcoef(y_test, pred_svm))

SVM AUC: 0.9031869078380708
SVM MCC: 0.6525059648658096


In [57]:
###### Predict MARINE peptides

marine_scores = svm.predict_proba(Xmar)[:,1]

In [58]:
###### Save results

df = pd.DataFrame({
    "peptide": marine,
    "ACP_score": marine_scores
})

df.to_csv("random_marine_acp_predictions_ESM2_physchem.csv", index=False)
df.head()

,peptide,ACP_score
0,AAEKEFIKYPYPTPLQYQQLATRLKVEKKLVRRW,0.850248
1,AAGTLYTYPENWR,0.792407
2,AAKKWAKAKWAKAKKWAKAA,0.886459
3,AAVPIVNLKDELLFPSWEALFSGSE,0.095057
4,AAWKWAWAKKWAKAKKWAKAA,0.774525
